In [2]:
#needed libraries 
import pandas as pd 
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from sklearn.preprocessing import OneHotEncoder
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "eda" / "src" / "dataset.py").exists():
    PROJECT_ROOT = Path.cwd().parent
EDA_DIR = PROJECT_ROOT / "eda"
if str(EDA_DIR) not in sys.path:
    sys.path.insert(0, str(EDA_DIR))

from src.dataset import load_split

In [3]:
# split the data from load split 

train_df = load_split("train")
val_df = load_split("val")
test_df = load_split("test")

for name, split in (("train", train_df), ("val", val_df), ("test", test_df)): 
    print(f'the split {name} has {len(split)} rows and {list(split.columns)}')

X_train, Y_train = train_df["x"], train_df["y"]
X_val, Y_val = val_df["x"], val_df["y_id"]
X_test, Y_test = test_df["x"], test_df["y_id"]


the split train has 2997217 rows and ['character_id', 'x', 'y', 'y_id']
the split val has 583098 rows and ['character_id', 'x', 'y', 'y_id']
the split test has 644398 rows and ['character_id', 'x', 'y', 'y_id']


In [4]:
CONTEXT_LGHT = 5 # most of the words had an extension of 5 char
MAX_TRAIN_ROWS = 30000
MAX_VAL_ROWS = 10000

#Train / validation samples 

train_sample = train_df.sample(
    n = min(MAX_TRAIN_ROWS, len(train_df)), random_state = 42
    )

validation_sample = val_df.sample(
    n = min(MAX_VAL_ROWS, len(val_df)), random_state = 42
    )

In [5]:
# context lenght to matrix to analisys 
def context_matrix (series, ctx_lenght): 
    characters = [list(text[-ctx_lenght:]) for text in series]
    return np.asarray(characters,dtype=object)

In [6]:
# Definition and standarization of variables 

X_train_ch = context_matrix(train_sample["x"], CONTEXT_LGHT)
X_val_ch = context_matrix(validation_sample["x"], CONTEXT_LGHT)
encoder = OneHotEncoder(handle_unknown="ignore")
X_train = encoder.fit_transform(X_train_ch)
X_val = encoder.transform(X_val_ch)
Y_train_sample = train_sample["y_id"].to_numpy()
Y_val_sample = validation_sample["y_id"].to_numpy()

In [7]:
# definition of the dictionary with a variety of configurations 

rf_config  = []
n_estimators = [100,200,400]
depth = [12,20,24]
min_leaf = [2,5]
max_feat =["sqrt",0.5]
class_weights = ["balanced", "balanced_subsample"]
#combination generation
for n in n_estimators:
    for d in depth:
        for l in min_leaf:
            for f in max_feat:
                for w in class_weights:

                    rf_config.append({
                        "name": f"rf_{n}_{d}_{l}_{f}_{w}",
                        "n_estimators": n,
                        "max_depth": d,
                        "min_samples_leaf": l,
                        "max_features": f,
                        "class_weight": w

                    })

print(f'Quantity of combinations {len(rf_config)}')


Quantity of combinations 72


In [8]:
#trainig and results 

experiment_results = []
trained_models = []

for cfg in rf_config: 
    params = {key: value for key, value in cfg.items() if key != "name"}
    model = RandomForestClassifier(**params, n_jobs=-1, random_state=42)
    model.fit(X_train, Y_train_sample)
    predicted = model.predict(X_val)

    experiment_results.append({
        "name": cfg["name"],
        "config": cfg.copy(),
        "accuracy": accuracy_score(Y_val_sample, predicted),
        "balanced_accuracy": balanced_accuracy_score(Y_val_sample, predicted),
        "macro_f1": f1_score(Y_val_sample, predicted, average="macro", zero_division=0),
        "weighted_f1": f1_score(Y_val_sample, predicted, average="weighted", zero_division=0),
    })
    trained_models.append({"name": cfg["name"], "model": model})
    print(f"{cfg['name']}: macro-F1={experiment_results[-1]['macro_f1']:.4f}, "
          f"accuracy={experiment_results[-1]['accuracy']:.4f}")


results_df = pd.DataFrame([
    {key: value for key, value in result.items() if key != "config"}
    for result in experiment_results
]).sort_values("macro_f1", ascending=False).reset_index(drop=True)



/home/juanest/Escritorio/machineLearning_ws1/ws/lib64/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


rf_100_12_2_sqrt_balanced: macro-F1=0.0957, accuracy=0.1825


/home/juanest/Escritorio/machineLearning_ws1/ws/lib64/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


rf_100_12_2_sqrt_balanced_subsample: macro-F1=0.1032, accuracy=0.2195


/home/juanest/Escritorio/machineLearning_ws1/ws/lib64/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


...

/home/juanest/Escritorio/machineLearning_ws1/ws/lib64/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


rf_400_24_5_sqrt_balanced_subsample: macro-F1=0.1201, accuracy=0.2495


/home/juanest/Escritorio/machineLearning_ws1/ws/lib64/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


rf_400_24_5_0.5_balanced: macro-F1=0.1165, accuracy=0.2015
rf_400_24_5_0.5_balanced_subsample: macro-F1=0.1319, accuracy=0.2732


/home/juanest/Escritorio/machineLearning_ws1/ws/lib64/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [9]:
results_df.head(5)

,name,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,rf_400_24_2_0.5_balanced_subsample,0.3098,0.197277,0.149928,0.354457
1,rf_200_24_2_0.5_balanced_subsample,0.3042,0.196897,0.147118,0.350117
2,rf_100_24_2_0.5_balanced_subsample,0.3025,0.194303,0.146028,0.347360
3,rf_400_24_2_sqrt_balanced_subsample,0.3159,0.202637,0.140170,0.361917
4,rf_200_24_2_sqrt_balanced_subsample,0.3098,0.200666,0.138771,0.356176
